<a href="https://colab.research.google.com/github/DiFedorchuk/ML_Course/blob/main/HW_2_2_%D0%9D%D0%B5%D0%B7%D0%B1%D0%B0%D0%BB%D0%B0%D0%BD%D1%81%D0%BE%D0%B2%D0%B0%D0%BD%D0%B0_%D0%B1%D0%B0%D0%B3%D0%B0%D1%82%D0%BE%D0%BA%D0%BB%D0%B0%D1%81%D0%BE%D0%B2%D0%B0_%D0%BA%D0%BB%D0%B0%D1%81%D0%B8%D1%84%D1%96%D0%BA%D0%B0%D1%86%D1%96%D1%8F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

У цьому ДЗ ми потренуємось розв'язувати задачу багатокласової класифікації за допомогою логістичної регресії з використанням стратегій One-vs-Rest та One-vs-One, оцінити якість моделей та порівняти стратегії.

### Опис задачі і даних

**Контекст**

В цьому ДЗ ми працюємо з даними про сегментацію клієнтів.

Сегментація клієнтів – це практика поділу бази клієнтів на групи індивідів, які схожі між собою за певними критеріями, що мають значення для маркетингу, такими як вік, стать, інтереси та звички у витратах.

Компанії, які використовують сегментацію клієнтів, виходять з того, що кожен клієнт є унікальним і що їхні маркетингові зусилля будуть більш ефективними, якщо вони орієнтуватимуться на конкретні, менші групи зі зверненнями, які ці споживачі вважатимуть доречними та які спонукатимуть їх до купівлі. Компанії також сподіваються отримати глибше розуміння уподобань та потреб своїх клієнтів з метою виявлення того, що кожен сегмент цінує найбільше, щоб точніше адаптувати маркетингові матеріали до цього сегменту.

**Зміст**.

Автомобільна компанія планує вийти на нові ринки зі своїми існуючими продуктами (P1, P2, P3, P4 і P5). Після інтенсивного маркетингового дослідження вони дійшли висновку, що поведінка нового ринку схожа на їхній існуючий ринок.

На своєму існуючому ринку команда з продажу класифікувала всіх клієнтів на 4 сегменти (A, B, C, D). Потім вони здійснювали сегментовані звернення та комунікацію з різними сегментами клієнтів. Ця стратегія працювала для них надзвичайно добре. Вони планують використати ту саму стратегію на нових ринках і визначили 2627 нових потенційних клієнтів.

Ви маєте допомогти менеджеру передбачити правильну групу для нових клієнтів.

В цьому ДЗ використовуємо дані `customer_segmentation_train.csv`[скачати дані](https://drive.google.com/file/d/1VU1y2EwaHkVfr5RZ1U4MPWjeflAusK3w/view?usp=sharing). Це `train.csv`з цього [змагання](https://www.kaggle.com/datasets/abisheksudarshan/customer-segmentation/data?select=train.csv)

**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
raw_df = pd.read_csv("/content/drive/MyDrive/Courses/Machine Learning/Data/customer_segmentation_train.csv", index_col=0)

In [15]:
raw_df.head()

,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
ID,,,,,,,,,,
462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,D
462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4,A
466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,B
461735,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6,B
462669,Female,Yes,40,Yes,Entertainment,NaN,High,6.0,Cat_6,A


In [24]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8068 entries, 462809 to 461879
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Gender           8068 non-null   object 
 1   Ever_Married     7928 non-null   object 
 2   Age              8068 non-null   int64  
 3   Graduated        7990 non-null   object 
 4   Profession       7944 non-null   object 
 5   Work_Experience  7239 non-null   float64
 6   Spending_Score   8068 non-null   object 
 7   Family_Size      7733 non-null   float64
 8   Var_1            7992 non-null   object 
 9   Segmentation     8068 non-null   object 
dtypes: float64(2), int64(1), object(7)
memory usage: 693.3+ KB


In [4]:
# Визначаємо колонки з нульовими значеннями
raw_df.columns[raw_df.isnull().any()]

Index(['Ever_Married', 'Graduated', 'Profession', 'Work_Experience',
       'Family_Size', 'Var_1'],
      dtype='object')

In [5]:
# Визначаємо їх відсоток
null_count = raw_df.isnull().sum()
null_percentage = round((raw_df.isnull().sum()/raw_df.shape[0])*100, 2)
null_percentage

,0
Gender,0.00
Ever_Married,1.74
Age,0.00
Graduated,0.97
Profession,1.54
Work_Experience,10.28
Spending_Score,0.00
Family_Size,4.15
Var_1,0.94
Segmentation,0.00


In [6]:
# Заповнюємо пусті значення медіаною в Work_Experience, Family_Size
work_experience = raw_df['Work_Experience'].median()
raw_df['Work_Experience'].fillna(work_experience, inplace=True)
family_size = raw_df['Family_Size'].median()
raw_df['Family_Size'].fillna(family_size, inplace=True)
# Заповнюємо модою пусті значення в Ever_Married, Graduated, Var_1
ever_married = raw_df['Ever_Married'].mode()[0]
raw_df['Ever_Married'].fillna(ever_married, inplace=True)
graduated = raw_df['Graduated'].mode()[0]
raw_df['Graduated'].fillna(graduated, inplace=True)
var_1 = raw_df['Var_1'].mode()[0]
raw_df['Var_1'].fillna(var_1, inplace=True)
profession = raw_df['Profession'].mode()[0]
raw_df['Profession'].fillna(profession, inplace=True)

/tmp/ipython-input-1412903260.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  raw_df['Work_Experience'].fillna(work_experience, inplace=True)
/tmp/ipython-input-1412903260.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace

In [7]:
# Перевірка чи є пусті значення
raw_df.columns[raw_df.isnull().any()]

Index([], dtype='object')

In [8]:
# Кодування колонок
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder
# Розділюємо на X та цільову змінну y
X = raw_df.drop('Segmentation', axis=1)
y = raw_df['Segmentation']
# Визначаємо числові та категоріальні стовпці
num_cols = ['Age', 'Work_Experience', 'Family_Size']
cat_cols = ['Gender', 'Ever_Married', 'Graduated', 'Profession', 'Var_1']
ord_cat_cols = ['Spending_Score']
# OrdinalEncoder кодуємо Spending_Score
ordinal_encoder = OrdinalEncoder(categories=[['Low', 'Average', 'High']])
X['Spending_Score_encoded'] = ordinal_encoder.fit_transform(X[['Spending_Score']])
X = X.drop('Spending_Score', axis=1)
# OneHotEncoder кодуємо cat_cols
onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
onehot_encoded_features = onehot_encoder.fit_transform(X[cat_cols])
onehot_feature_names = onehot_encoder.get_feature_names_out(cat_cols)
onehot_df = pd.DataFrame(onehot_encoded_features, columns=onehot_feature_names, index=X.index)
# Об'єднуємо
X_processed = pd.concat([X[num_cols], X['Spending_Score_encoded'], onehot_df], axis=1)
# LabelEncoder кодуємо Segmentation
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [12]:
# Масштабуємо числові колонки
from sklearn.preprocessing import StandardScaler
X_scaled = X_processed.copy()
scaler = StandardScaler()
X_scaled[num_cols] = scaler.fit_transform(X_scaled[num_cols])
X_processed = X_scaled.copy()

In [14]:
# Розділюємо на тренувальний та тестовий набори
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_processed, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
print(f"Розмір X_train: {X_train.shape}")
print(f"Розмір X_test: {X_test.shape}")
print(f"Розмір y_train: {y_train.shape}")
print(f"Розмір y_test: {y_test.shape}")

Розмір X_train: (6454, 26)
Розмір X_test: (1614, 26)
Розмір y_train: (6454,)
Розмір y_test: (1614,)


**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

In [27]:
# Визначаємо індекси категоріальних ознак
num_cols = ['Age', 'Work_Experience', 'Family_Size']
categorical_feature_indices = []
for i, col in enumerate(X_train.columns):
    if col not in num_cols:
        categorical_feature_indices.append(i)
print(categorical_feature_indices)

[3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]


In [28]:
# Використаємо SMOTENC
from imblearn.over_sampling import SMOTENC
smotenc = SMOTENC(categorical_features=categorical_feature_indices, random_state=42)
X_train_smotenc, y_train_smotenc = smotenc.fit_resample(X_train, y_train)
print(f"Оригінальний X_train розмір: {X_train.shape}, y_train розмір: {y_train.shape}")
print(f"SMOTENC X_train розмір {X_train_smotenc.shape}, y_train розмір: {y_train_smotenc.shape}")

# Використаємо SMOTETomek
from imblearn.combine import SMOTETomek
smotetomek = SMOTETomek(random_state=42)
X_train_smotetomek, y_train_smotetomek = smotetomek.fit_resample(X_train, y_train)
print(f"SMOTETomek X_train розмір: {X_train_smotetomek.shape}, y_train розмір: {y_train_smotetomek.shape}")

Оригінальний X_train розмір: (6454, 26), y_train розмір: (6454,)
SMOTENC X_train розмір (7256, 26), y_train розмір: (7256,)
SMOTETomek X_train розмір: (5892, 26), y_train розмір: (5892,)


**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

In [30]:
# 1
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
# Навчаємо модель на оригінальних даних
original_model = OneVsRestClassifier(LogisticRegression(solver='liblinear', random_state=42))
original_model.fit(X_train, y_train)

# Навчаємо модель на SMOTENC
smotenc_model = OneVsRestClassifier(LogisticRegression(solver='liblinear', random_state=42))
smotenc_model.fit(X_train_smotenc, y_train_smotenc)

# Навчаємо модель на SMOTETomek
smotetomek_model = OneVsRestClassifier(LogisticRegression(solver='liblinear', random_state=42))
smotetomek_model.fit(X_train_smotetomek, y_train_smotetomek)

OneVsRestClassifier(estimator=LogisticRegression(random_state=42,
                                                 solver='liblinear'))

In [32]:
# 2
from sklearn.metrics import classification_report
# Classification Report для оригінальної моделі
y_pred_original = original_model.predict(X_test)
print("\n Classification Report для оригінальної моделі: ")
print(classification_report(y_test, y_pred_original, target_names=label_encoder.classes_))

# Classification Report для SMOTENC моделі
y_pred_smotenc = smotenc_model.predict(X_test)
print("\nClassification Report для SMOTENC моделі: ")
print(classification_report(y_test, y_pred_smotenc, target_names=label_encoder.classes_))

# Classification Report для SMOTETomek моделі
y_pred_smotetomek = smotetomek_model.predict(X_test)
print("\nClassification Report для SMOTETomek моделі: ")
print(classification_report(y_test, y_pred_smotetomek, target_names=label_encoder.classes_))


 Classification Report для оригінальної моделі: 
              precision    recall  f1-score   support

           A       0.41      0.45      0.43       394
           B       0.43      0.15      0.23       372
           C       0.49      0.64      0.55       394
           D       0.64      0.75      0.69       454

    accuracy                           0.51      1614
   macro avg       0.49      0.50      0.48      1614
weighted avg       0.50      0.51      0.49      1614


Classification Report для SMOTENC моделі: 
              precision    recall  f1-score   support

           A       0.42      0.47      0.44       394
           B       0.43      0.23      0.30       372
           C       0.50      0.62      0.55       394
           D       0.66      0.71      0.69       454

    accuracy                           0.52      1614
   macro avg       0.50      0.51      0.50      1614
weighted avg       0.51      0.52      0.51      1614


Classification Report для SMOTETome

3. Для оцінки якості моделей потрібно звертати увагу на macro avg F1, так як ми маємо дизбаланс чотирьох класів та багатокласову класифікацію
4. Бачимо що найбільший показник має macro avg F1 має SMOTENC 0,50 , він трохи більший за SMOTETomek
5. Так, ми бачимо що показники дуже близькі між собою, можливо це залежить від самого датасету, можливо в нас не достатньо даних, або вони не дуже інформативні, також можливо потрібно використати  якусь іншу не лінійну модель